# Engineer causal aircraft-rotation features from full BTS history

Extend the shared departure feature table along a separate, append-only experiment path. For each target flight, the scheduled-departure timestamp is the prediction cutoff. The aircraft tail number is matched against every eligible movement in the raw airport BTS file, including origins outside the project's modeling-airport cohort. A preceding arrival is a rotation match; a preceding non-cancelled airport departure blocks older, stale arrivals.

The inbound leg's realized arrival delay and actual turn time are exposed only when that aircraft had arrived by the target cutoff. If it had not arrived, the dataset records only the observable not-arrived state and scheduled overdue duration. The output schema matches the cohort-limited rotation dataset so Experiment 06 can isolate history scope without changing the model manifest.

In [1]:
YEAR = 2019

AIRPORT = "JFK"

In [2]:
# Parameters
YEAR = 2024
AIRPORT = "JFK"


## Configure the departure and full-history sources

The target rows come from the unchanged shared departure feature file. Rotation history comes from the raw BTS airport file so movements outside the working-airport cohort can still establish the immediately preceding tail event. The output is separately named so existing feature datasets and experiment results remain intact.

In [3]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import MODEL_TARGETS
from feature_engineering_rotation import (
    ROTATION_AUDIT_COLUMNS,
    ROTATION_FEATURES,
    ROTATION_OUTPUT_COLUMNS,
    add_departure_rotation_features_full_history,
    validate_departure_rotation_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
DEPARTURE_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures.csv"
RAW_HISTORY_FILE = PROJECT_ROOT / f"data/bts/raw/{YEAR}/{AIRPORT}.csv"
OUTPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures_rotation_full_history.csv"

print(pd.Series({
    "departures": str(DEPARTURE_FILE),
    "full raw history": str(RAW_HISTORY_FILE),
    "output": str(OUTPUT_FILE),
}))

departures          /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
full raw history    /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
output              /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
dtype: str


## Load and validate the target and raw-history populations

The notebook validates rather than filters the target departures. The raw BTS source must cover the requested year and contain movements arriving at or departing from the configured airport; the shared helper then keeps completed inbound events and non-cancelled outbound blockers.

In [4]:
for source_file in [DEPARTURE_FILE, RAW_HISTORY_FILE]:
    if not source_file.is_file():
        raise FileNotFoundError(f"Required rotation source does not exist: {source_file}")

departures = pd.read_csv(DEPARTURE_FILE, low_memory=False)
raw_history = pd.read_csv(RAW_HISTORY_FILE, low_memory=False)

departure_required = {"Year", "Origin", "DATE", "Tail_Number", MODEL_TARGETS["1A"]}
history_required = {
    "Year", "FlightDate", "Reporting_Airline", "Tail_Number",
    "Flight_Number_Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    "ArrDelay", "ArrDel15", "Cancelled", "Diverted",
}
if departure_required - set(departures.columns):
    raise KeyError(f"Departure feature data is missing: {sorted(departure_required - set(departures.columns))}")
if history_required - set(raw_history.columns):
    raise KeyError(f"Raw BTS history is missing: {sorted(history_required - set(raw_history.columns))}")
if set(ROTATION_OUTPUT_COLUMNS) & set(departures.columns):
    raise ValueError("Input already contains rotation fields; use the base departure feature file")

departure_origin = departures["Origin"].astype("string").str.strip().str.upper()
history_origin = raw_history["Origin"].astype("string").str.strip().str.upper()
history_destination = raw_history["Dest"].astype("string").str.strip().str.upper()
departure_year = pd.to_numeric(departures["Year"], errors="coerce")
history_year = pd.to_numeric(raw_history["Year"], errors="coerce")
if not departure_origin.eq(AIRPORT).all():
    raise ValueError(f"Departure input contains origins other than {AIRPORT}")
if not (history_origin.eq(AIRPORT) | history_destination.eq(AIRPORT)).all():
    raise ValueError(f"Raw history contains movements unrelated to {AIRPORT}")
if not departure_year.eq(YEAR).all() or not history_year.eq(YEAR).all():
    raise ValueError(f"A rotation input contains years other than {YEAR}")

target = pd.to_numeric(departures[MODEL_TARGETS["1A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("DepDel15 must be complete and binary")

print(pd.Series({
    "departure rows": len(departures),
    "departure columns": len(departures.columns),
    "raw history rows": len(raw_history),
    "raw history columns": len(raw_history.columns),
    "delayed departures": int(target.sum()),
}))

departure rows         104715
departure columns         112
raw history rows       246682
raw history columns       110
delayed departures      21344
dtype: int64


## Add full-history rotation features

Airport-local schedule clocks are converted to UTC before leg ordering. Completed inbound movements from every raw origin and all non-cancelled outbound blockers participate in tail history. Scheduled inbound arrival dates are reconstructed by choosing the destination-local date whose UTC elapsed time best agrees with BTS `CRSElapsedTime`; the residual remains available for audit.

In [5]:
features = add_departure_rotation_features_full_history(
    departures, raw_history, airport=AIRPORT
)
if len(features) != len(departures):
    raise ValueError("Rotation feature engineering changed the departure row count")
if not features[departures.columns].equals(departures):
    raise ValueError("Rotation feature engineering changed one or more source columns")
if list(features.columns[-len(ROTATION_OUTPUT_COLUMNS):]) != ROTATION_OUTPUT_COLUMNS:
    raise ValueError("Rotation output columns were not appended in the documented order")

feature_validation = validate_departure_rotation_features(features)
feature_validation

,dtype,missing_count,missing_percent,minimum,maximum
ROTATION_ACTUAL_TURN_MINUTES,float64,7374,7.041971,0.000000,462199.000000
ROTATION_INBOUND_ARRIVED_BY_CUTOFF,int8,0,0.000000,0.000000,1.000000
ROTATION_INBOUND_ARR_DELAY,float64,7374,7.041971,-69.000000,2972.000000
ROTATION_INBOUND_DELAYED_15,float64,7374,7.041971,0.000000,1.000000
ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF,int8,0,0.000000,0.000000,1.000000
ROTATION_INBOUND_ORIGIN,str,4261,4.069140,NaN,NaN
ROTATION_INBOUND_OVERDUE_MINUTES,float64,4261,4.069140,0.000000,1568.000000
ROTATION_LOG_ACTUAL_TURN_MINUTES,float64,7374,7.041971,0.000000,13.043753
ROTATION_LOG_INBOUND_OVERDUE_MINUTES,float64,4261,4.069140,0.000000,7.358194
ROTATION_LOG_SCHEDULED_TURN_MINUTES,float64,4261,4.069140,0.693147,13.043714


## Review match coverage and information content

The target-rate summary is descriptive only; it does not select features or tune a model. A large rate for `NOT_ARRIVED` is plausible because that state directly means the assigned aircraft is not at the gate by scheduled departure. This information is causal at the cutoff but requires a timely aircraft-assignment and arrival-event feed in deployment.

In [6]:
status_summary = (
    features.assign(_TARGET=target)
    .groupby("ROTATION_STATUS", dropna=False, observed=True)
    .agg(rows=("_TARGET", "size"), delayed=("_TARGET", "sum"), delay_rate=("_TARGET", "mean"))
    .sort_values("rows", ascending=False)
)
status_summary["row_percent"] = status_summary["rows"] / len(features) * 100
status_summary

,rows,delayed,delay_rate,row_percent
ROTATION_STATUS,,,,
ARRIVED,97341,16237.0,0.166805,92.958029
PREVIOUS_EVENT_DEPARTURE,3855,1902.0,0.493385,3.681421
NOT_ARRIVED,3113,3106.0,0.997751,2.972831
NO_PRIOR_EVENT,406,99.0,0.243842,0.387719


In [7]:
timing_audit = pd.Series({
    "rotation matches": int(features["ROTATION_MATCH_FOUND"].sum()),
    "arrived by cutoff": int(features["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"].sum()),
    "not arrived by cutoff": int(features["ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF"].sum()),
    "actual outcomes visible before cutoff violations": int(
        features.loc[
            features["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"].eq(0),
            ["ROTATION_INBOUND_ARR_DELAY", "ROTATION_INBOUND_DELAYED_15",
             "ROTATION_ACTUAL_TURN_MINUTES", "ROTATION_PRIOR_ACTUAL_ARRIVAL_UTC"],
        ].notna().any(axis=1).sum()
    ),
    "schedule reconstructions with residual over 1 minute": int(
        pd.to_numeric(
            features["ROTATION_SCHEDULE_RECONSTRUCTION_ERROR_MINUTES"],
            errors="coerce",
        ).gt(1).sum()
    ),
}, name="rotation timing validation")
timing_audit

rotation matches                                        100454
arrived by cutoff                                        97341
not arrived by cutoff                                     3113
actual outcomes visible before cutoff violations             0
schedule reconstructions with residual over 1 minute         2
Name: rotation timing validation, dtype: int64

## Save the full-history rotation-enhanced departure dataset

Retain every source and audit column, append the rotation fields using the same schema as the cohort-limited path, and write a separately named annual dataset. All established departure, backlog, and rotation files remain intact.

In [8]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT,
    "year": YEAR,
    "rows": len(features),
    "columns": len(features.columns),
    "rotation features": len(ROTATION_FEATURES),
    "rotation audit columns": len(ROTATION_AUDIT_COLUMNS),
    "history scope": "all eligible raw airport movements",
    "target": MODEL_TARGETS["1A"],
    "output": str(OUTPUT_FILE),
}, name="departure rotation feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary

Saved 104,715 rows to /Users/johnkyte/Projects/berkeley_ml_and_ai/capstone/data/features/JFK_2024_departures_rotation_full_history.csv


airport                                                                 JFK
year                                                                   2024
rows                                                                 104715
columns                                                                 133
rotation features                                                        13
rotation audit columns                                                    8
history scope                            all eligible raw airport movements
target                                                             DepDel15
output                    /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
Name: departure rotation feature summary, dtype: object